# Single Pendulum: Advantage Actor-Critic (A2C)

A complete policy-based baseline with two explicit learning blocks: a critic that learns
state values and an actor that learns a continuous squashed-Gaussian policy. The plant is
identical to the Q-learning and DQN notebooks. Run a smoke test before tuning long runs.


## 1. Runtime setup


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive")

IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_COLAB or IN_KAGGLE:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--upgrade", "-q",
        "jax[cuda12]", "mujoco>=3.2", "gymnasium>=1.0", "flax==0.12.8", "optax==0.2.8",
        "pandas>=2.0", "matplotlib>=3.8", "imageio>=2.34", "imageio-ffmpeg>=0.5",
    ], check=True)

# Keep learning arrays resident on the accelerator and use EGL for GPU-backed video rendering.
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "true")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.85")

import flax
import jax
import jax.numpy as jnp
import optax

gpu_devices = [device for device in jax.devices() if device.platform == "gpu"]
print(f"JAX {jax.__version__} | Flax {flax.__version__} | Optax {optax.__version__}")
print("JAX backend:", jax.default_backend(), "| devices:", jax.devices())
if (IN_COLAB or IN_KAGGLE) and not gpu_devices:
    raise RuntimeError(
        "GPU accelerator required: enable a GPU in the Colab or Kaggle notebook settings, "
        "restart the runtime, and run all cells again."
    )
if (IN_COLAB or IN_KAGGLE) and flax.__version__ != "0.12.8":
    raise RuntimeError("Restart the notebook runtime, then run all cells again.")

accelerator_probe = jax.jit(lambda x: x @ x)(jnp.ones((64, 64), dtype=jnp.float32))
accelerator_probe.block_until_ready()
probe_device = next(iter(accelerator_probe.devices()))
if (IN_COLAB or IN_KAGGLE) and probe_device.platform != "gpu":
    raise RuntimeError("GPU accelerator required, but the JIT probe ran on " + str(probe_device))
print("JIT accelerator probe:", probe_device)


## 2. Tuning and persistent paths


In [ ]:
OUTPUT_DIR = Path("/content/drive/MyDrive/ProjectsRuns/TIPy/runs/single/a2c/run-001") if IN_COLAB else Path.cwd() / "a2c-run"
CHECKPOINT_DIR, METRICS_PATH = OUTPUT_DIR / "checkpoints", OUTPUT_DIR / "metrics.csv"
DASHBOARD_PATH = OUTPUT_DIR / "dashboard.png"
SEED, TOTAL_STEPS, MAX_EPISODE_STEPS, ACTION_LIMIT = 42, 200_000, 2000, 100.0
GAMMA, N_STEPS, ACTOR_LR, CRITIC_LR, ENTROPY_COEF = .99, 512, 3e-4, 1e-3, .001
CHECKPOINT_EVERY, SMOKE_TEST = 25, False
if SMOKE_TEST: TOTAL_STEPS, MAX_EPISODE_STEPS, N_STEPS, CHECKPOINT_EVERY = 512, 128, 32, 1
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Continuous-action plant
The actor outputs normalized force in `[-1,1]`; the environment scales it to 100 N.


In [ ]:
import gymnasium as gym
from gymnasium import spaces
import mujoco
import numpy as np

MODEL_XML = r"""
<mujoco model="cartpole_single">
  <compiler angle="radian" autolimits="true" inertiafromgeom="false" />
  <option timestep="0.005" gravity="0 0 -9.81" integrator="RK4" />
  <default>
    <geom contype="0" conaffinity="0" />
  </default>
  <visual>
    <headlight diffuse="0.7 0.7 0.7" ambient="0.3 0.3 0.3" specular="0.1 0.1 0.1" />
    <rgba haze="0.15 0.2 0.25 1" />
  </visual>
  <asset>
    <material name="floor_mat" rgba="0.16 0.22 0.28 1" />
    <material name="metal_mat" rgba="0.65 0.68 0.72 1" />
    <material name="cart_mat" rgba="0.82 0.18 0.18 1" />
    <material name="pole_mat" rgba="0.2 0.75 0.3 1" />
    <material name="site_mat" rgba="0.95 0.9 0.15 0.9" />
    <material name="rail_limit_mat" rgba="0.95 0.55 0.05 1" />
  </asset>
  <worldbody>
    <camera name="spectate" pos="0 3 1.4" fovy="90" xyaxes="-1 0 0 0 -0.1240 0.9923" />
    <light diffuse="0.7 0.7 0.7" pos="0 0 3.5" dir="0 0 -1" />
    <geom name="floor" type="plane" size="5 5 0.1" material="floor_mat" />
    <body name="frame">
      <geom name="rail" type="box" pos="0 0 0.85" size="2.2 0.05 0.05" material="metal_mat" />
      <geom name="rail_limit_left" type="box" pos="-2.2 0 0.95" size="0.025 0.12 0.1" material="rail_limit_mat" />
      <geom name="rail_limit_right" type="box" pos="2.2 0 0.95" size="0.025 0.12 0.1" material="rail_limit_mat" />
    </body>
    <body name="cart" pos="0 0 1">
      <joint name="cart_slide" type="slide" axis="1 0 0" range="-2.2 2.2" frictionloss="0.02" damping="0.1" />
      <inertial pos="0 0 0" mass="2" diaginertia="0.0333 0.0333 0.0333" />
      <geom name="cart_geom" type="box" size="0.125 0.08 0.1" mass="2.0" material="cart_mat" />
      <site name="cart_center_site" pos="0 0 0" size="0.02" type="sphere" material="site_mat" />
      <geom name="mount_pin" type="capsule" pos="0 0.095 0" axisangle="1 0 0 1.5708" size="0.015 0.03" material="metal_mat" />
      <body name="pole" pos="0 0.11 0" quat="6.12323399574e-17 0 -1 0">
        <joint name="pole_hinge" type="hinge" axis="0 -1 0" frictionloss="0.01" damping="0.03" ref="3.1415926535897931" limited="false" />
        <inertial pos="0 0 0.3" mass="0.7" diaginertia="0.0210233333333 0.0210933333333 0.000116666666667" />
        <site name="pole_hinge_site" pos="0 0 0" size="0.015" type="sphere" material="site_mat" />
        <geom name="pole_geom" type="box" pos="0 0 0.3" size="0.02 0.01 0.3" mass="0.7" material="pole_mat" />
        <site name="pole_tip_site" pos="0 0 0.6" size="0.015" type="sphere" material="site_mat" />
      </body>
    </body>
    <camera name="replay" pos="0 6 1.4" fovy="50" xyaxes="-1 0 0 0 -0.15 0.988686" />
  </worldbody>
  <sensor>
    <jointpos name="cart_position" joint="cart_slide" />
    <jointpos name="pole1_relative_angle" joint="pole_hinge" />
    <jointvel name="cart_velocity" joint="cart_slide" />
    <jointvel name="pole1_relative_velocity" joint="pole_hinge" />
  </sensor>
  <actuator>
    <motor name="cart_motor" joint="cart_slide" gear="1" ctrlrange="-100 100" forcerange="-100 100" />
  </actuator>
</mujoco>
"""

def normalize_angle(theta):
    return float((theta + np.pi) % (2.0 * np.pi) - np.pi)

class SinglePendulumEnv(gym.Env):
    """Self-contained MuJoCo cart-pole swing-up environment."""
    def __init__(self, discrete=True, max_episode_steps=2000, action_limit=100.0):
        super().__init__()
        self.model = mujoco.MjModel.from_xml_string(MODEL_XML)
        self.data = mujoco.MjData(self.model)
        self.discrete = discrete
        self.max_episode_steps = max_episode_steps
        self.action_limit = float(action_limit)
        self.rail_limit = 2.2
        self.dt = float(self.model.opt.timestep)
        self.current_step = 0
        self.observation_space = spaces.Box(-1.0, 1.0, shape=(5,), dtype=np.float32)
        if discrete:
            self.action_table = np.linspace(-self.action_limit, self.action_limit, 7)
            self.action_space = spaces.Discrete(7)
        else:
            self.action_space = spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float32)

    def _observation(self):
        x, theta = map(float, self.data.qpos)
        dx, dtheta = map(float, self.data.qvel)
        theta = normalize_angle(theta)
        return np.array([
            np.clip(x / self.rail_limit, -1.0, 1.0),
            np.clip(dx / 5.0, -1.0, 1.0),
            np.cos(theta), np.sin(theta),
            np.clip(dtheta / 15.0, -1.0, 1.0),
        ], dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[:] = [self.np_random.uniform(-0.02, 0.02),
                             np.pi + self.np_random.uniform(-0.05, 0.05)]
        self.data.qvel[:] = self.np_random.uniform([-0.01, -0.02], [0.01, 0.02])
        mujoco.mj_forward(self.model, self.data)
        return self._observation(), {}

    def step(self, action):
        self.current_step += 1
        if self.discrete:
            force = float(self.action_table[int(action)])
        else:
            force = float(np.asarray(action).reshape(-1)[0]) * self.action_limit
        force = float(np.clip(force, -self.action_limit, self.action_limit))
        self.data.ctrl[0] = force
        mujoco.mj_step(self.model, self.data)
        x, theta = map(float, self.data.qpos)
        dx, dtheta = map(float, self.data.qvel)
        theta = normalize_angle(theta)
        reward = (np.cos(theta) - 0.2 * (x / self.rail_limit) ** 2
                  - 0.01 * dx ** 2 - 0.005 * dtheta ** 2
                  - 0.001 * (force / self.action_limit) ** 2
                  + (2.0 if abs(theta) < 0.35 else 0.0))
        terminated = bool(abs(x) >= self.rail_limit)
        if terminated:
            reward -= 10.0
        truncated = bool(self.current_step >= self.max_episode_steps)
        info = {"x": x, "theta": theta, "dx": dx, "dtheta": dtheta, "force": force}
        return self._observation(), float(reward), terminated, truncated, info

# Contract and reproducibility checks.
_env = SinglePendulumEnv(max_episode_steps=10)
_a, _ = _env.reset(seed=7)
_b, _ = _env.reset(seed=7)
assert _a.shape == (5,) and np.allclose(_a, _b)
assert _env.action_space.n == 7 and np.all(np.isfinite(_a))
print("Environment check passed; initial observation:", _a)


## 4. Critic: value-based block

The critic estimates `V(s)`, the discounted return expected from a state. It is trained by
regression toward n-step bootstrapped returns. This is the value-based half of actor-critic.


In [ ]:
import flax.linen as nn
import jax.numpy as jnp
import optax

class Critic(nn.Module):
    @nn.compact
    def __call__(self, obs):
        x = nn.tanh(nn.Dense(128)(obs)); x = nn.tanh(nn.Dense(128)(x))
        return nn.Dense(1)(x).squeeze(-1)


## 5. Actor: policy-based block

The actor predicts a Gaussian in unconstrained space. `tanh` maps samples into the valid
action range. The log probability includes the tanh Jacobian correction, so it describes
the action actually sent to the environment rather than an unclipped Normal sample.


In [ ]:
class Actor(nn.Module):
    @nn.compact
    def __call__(self, obs):
        x = nn.tanh(nn.Dense(128)(obs)); x = nn.tanh(nn.Dense(128)(x))
        mean = nn.Dense(1)(x)
        log_std = self.param("log_std", nn.initializers.constant(-.5), (1,))
        return mean, jnp.clip(jnp.broadcast_to(log_std, mean.shape), -5, 2)

def gaussian_log_prob(mean, log_std, latent):
    log_normal = -.5 * (((latent-mean)/jnp.exp(log_std))**2 + 2*log_std + jnp.log(2*jnp.pi))
    action = jnp.tanh(latent)
    return jnp.sum(log_normal - jnp.log(1-action**2+1e-6), axis=-1)


## 6. A2C learner
Returns bootstrap at rollout boundaries and time limits, but not at true rail terminals.


In [ ]:
class A2CAgent:
    def __init__(self, obs_dim, seed):
        self.actor, self.critic = Actor(), Critic(); key1, key2 = jax.random.split(jax.random.PRNGKey(seed))
        dummy = jnp.zeros((1, obs_dim)); self.actor_params = self.actor.init(key1, dummy)
        self.critic_params = self.critic.init(key2, dummy)
        self.actor_opt, self.critic_opt = optax.adam(ACTOR_LR), optax.adam(CRITIC_LR)
        self.actor_state = self.actor_opt.init(self.actor_params); self.critic_state = self.critic_opt.init(self.critic_params)
        self.key, self.total_steps = jax.random.PRNGKey(seed + 1), 0
        self._update = jax.jit(self._update_impl)

    def act(self, obs, deterministic=False):
        mean, log_std = self.actor.apply(self.actor_params, jnp.asarray(obs)[None])
        self.key, sample_key = jax.random.split(self.key)
        latent = mean if deterministic else mean + jnp.exp(log_std) * jax.random.normal(sample_key, mean.shape)
        return np.asarray(jnp.tanh(latent)[0]), np.asarray(latent[0])

    def value(self, obs): return float(self.critic.apply(self.critic_params, jnp.asarray(obs)[None])[0])

    def _update_impl(self, actor_params, critic_params, actor_state, critic_state, batch):
        values = self.critic.apply(critic_params, batch["obs"])
        advantages = batch["returns"] - values
        normalized = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        def actor_loss_fn(params):
            mean, log_std = self.actor.apply(params, batch["obs"])
            log_prob = gaussian_log_prob(mean, log_std, batch["latent"])
            entropy = -jnp.mean(log_prob)
            return -jnp.mean(jax.lax.stop_gradient(normalized) * log_prob) - ENTROPY_COEF * entropy, entropy
        (actor_loss, entropy), actor_grads = jax.value_and_grad(actor_loss_fn, has_aux=True)(actor_params)
        def critic_loss_fn(params):
            prediction = self.critic.apply(params, batch["obs"])
            return .5 * jnp.mean((prediction - batch["returns"])**2)
        critic_loss, critic_grads = jax.value_and_grad(critic_loss_fn)(critic_params)
        updates, actor_state = self.actor_opt.update(actor_grads, actor_state, actor_params)
        actor_params = optax.apply_updates(actor_params, updates)
        updates, critic_state = self.critic_opt.update(critic_grads, critic_state, critic_params)
        critic_params = optax.apply_updates(critic_params, updates)
        return actor_params, critic_params, actor_state, critic_state, actor_loss, critic_loss, entropy

    def update(self, obs, latent, rewards, terminals, bootstrap):
        returns, running = [], float(bootstrap)
        for reward, terminal in zip(reversed(rewards), reversed(terminals)):
            running = float(reward) + GAMMA * running * (1-float(terminal)); returns.append(running)
        batch = {"obs": jnp.asarray(obs), "latent": jnp.asarray(latent), "returns": jnp.asarray(returns[::-1])}
        result = self._update(self.actor_params, self.critic_params, self.actor_state, self.critic_state, batch)
        self.actor_params, self.critic_params, self.actor_state, self.critic_state = result[:4]
        return tuple(float(x) for x in result[4:])

    def state_dict(self):
        return jax.device_get({"actor": self.actor_params, "critic": self.critic_params,
            "actor_state": self.actor_state, "critic_state": self.critic_state,
            "key": self.key, "total_steps": self.total_steps})

    def load_state_dict(self, state):
        self.actor_params = jax.tree.map(jnp.asarray, state["actor"]); self.critic_params = jax.tree.map(jnp.asarray, state["critic"])
        self.actor_state = jax.tree.map(jnp.asarray, state["actor_state"]); self.critic_state = jax.tree.map(jnp.asarray, state["critic_state"])
        self.key, self.total_steps = jnp.asarray(state["key"]), state["total_steps"]


## 7. Persistence and metrics


In [ ]:
import csv, os, pickle, time
from datetime import datetime, timezone
FIELDS = ["timestamp","episode","total_steps","reward","actor_loss","critic_loss","entropy","episode_length","steps_per_second","success"]
def save_state(agent, episode):
    path=CHECKPOINT_DIR/f"checkpoint_{agent.total_steps:012d}.pkl"; temp=path.with_suffix(".tmp")
    with temp.open("wb") as file:
        pickle.dump({"version":1,"episode":episode,"agent":agent.state_dict()},file,pickle.HIGHEST_PROTOCOL); file.flush(); os.fsync(file.fileno())
    temp.replace(path)
    for old in sorted(CHECKPOINT_DIR.glob("checkpoint_*.pkl"))[:-3]: old.unlink()
    return path
def restore_state(agent):
    for path in sorted(CHECKPOINT_DIR.glob("checkpoint_*.pkl"),reverse=True):
        try:
            with path.open("rb") as file: state=pickle.load(file)
            agent.load_state_dict(state["agent"]); print("Resumed",path.name); return state["episode"]
        except Exception as error: print("Skipped",path.name,error)
    return 0
def log_row(row):
    new=not METRICS_PATH.exists()
    with METRICS_PATH.open("a",newline="") as file:
        writer=csv.DictWriter(file,fieldnames=FIELDS)
        if new: writer.writeheader()
        writer.writerow(row); file.flush(); os.fsync(file.fileno())


## 8. Train
Updates occur every `N_STEPS` or at episode end. Episode metrics aggregate all updates.


In [ ]:
env=SinglePendulumEnv(False,MAX_EPISODE_STEPS,ACTION_LIMIT); agent=A2CAgent(5,SEED)
start_episode=restore_state(agent); clock=time.perf_counter(); start_steps=agent.total_steps; episode=start_episode
try:
    while agent.total_steps<TOTAL_STEPS:
        episode+=1; observation,_=env.reset(seed=SEED+episode); reward_sum=0.; success=False
        update_metrics=[]; rollout={"obs":[],"latent":[],"rewards":[],"terminals":[]}
        for length in range(1,MAX_EPISODE_STEPS+1):
            action,latent=agent.act(observation); next_obs,reward,terminated,truncated,info=env.step(action)
            rollout["obs"].append(observation); rollout["latent"].append(latent)
            rollout["rewards"].append(reward); rollout["terminals"].append(terminated)
            observation=next_obs; reward_sum+=reward; agent.total_steps+=1; success|=abs(info["theta"])<.35
            boundary=len(rollout["rewards"])>=N_STEPS or terminated or truncated or agent.total_steps>=TOTAL_STEPS
            if boundary:
                bootstrap=0. if terminated else agent.value(observation)
                update_metrics.append(agent.update(**rollout,bootstrap=bootstrap))
                rollout={"obs":[],"latent":[],"rewards":[],"terminals":[]}
            if terminated or truncated or agent.total_steps>=TOTAL_STEPS: break
        actor_loss,critic_loss,entropy=np.mean(update_metrics,axis=0)
        elapsed=max(time.perf_counter()-clock,1e-9)
        log_row({"timestamp":datetime.now(timezone.utc).isoformat(),"episode":episode,"total_steps":agent.total_steps,
                 "reward":reward_sum,"actor_loss":actor_loss,"critic_loss":critic_loss,"entropy":entropy,
                 "episode_length":length,"steps_per_second":(agent.total_steps-start_steps)/elapsed,"success":int(success)})
        if episode%CHECKPOINT_EVERY==0: print("Saved",save_state(agent,episode))
        if episode%10==0: print(episode,agent.total_steps,round(reward_sum,1))
finally: save_state(agent,episode)


## 9. Static dashboard


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
metrics = pd.read_csv(METRICS_PATH).drop_duplicates("episode", keep="last").sort_values("episode")
window = min(20, len(metrics)); fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
fig.suptitle("TIPy Single Pendulum A2C", fontsize=16, fontweight="bold")
axes[0,0].plot(metrics.episode, metrics.reward, alpha=.3)
axes[0,0].plot(metrics.episode, metrics.reward.rolling(window,min_periods=1).mean()); axes[0,0].set_title("Reward")
axes[0,1].plot(metrics.episode, metrics.actor_loss); axes[0,1].set_title("Actor loss")
axes[0,2].plot(metrics.episode, metrics.critic_loss); axes[0,2].set_title("Critic loss")
axes[1,0].plot(metrics.episode, metrics.entropy); axes[1,0].set_title("Policy entropy estimate")
axes[1,1].plot(metrics.episode, metrics.episode_length); axes[1,1].set_title("Episode length")
axes[1,2].plot(metrics.episode, metrics.steps_per_second); axes[1,2].set_title("Steps/second")
for axis in axes.flat: axis.set_xlabel("Episode"); axis.grid(alpha=.25)
fig.savefig(DASHBOARD_PATH, dpi=160); plt.show(); print("Saved", DASHBOARD_PATH)


## 10. Deterministic evaluation
Evaluation uses the actor mean and performs no learning.


In [ ]:
rewards=[]
for episode in range(10):
    obs,_=env.reset(seed=30_000+episode); total=0.
    while True:
        action,_=agent.act(obs,deterministic=True); obs,reward,terminated,truncated,info=env.step(action); total+=reward
        if terminated or truncated: break
    rewards.append(total)
print("Deterministic reward mean/std:",np.mean(rewards),np.std(rewards))


## Replay The Trained Run

Colab cannot reliably open MuJoCo's interactive desktop viewer. This block runs a
deterministic evaluation, streams rendered frames directly into an MP4, saves it in
the run directory, and displays it inline. It replays the current trained policy or
controller; it is not an exact recording of a stochastic training episode.


In [ ]:
import imageio.v2 as imageio
from IPython.display import Video, display

REPLAY_SEED, REPLAY_SECONDS, REPLAY_FPS = 50_003, 10.0, 50
REPLAY_PATH = OUTPUT_DIR / "replay.mp4"
replay_env = SinglePendulumEnv(False, int(REPLAY_SECONDS / 0.005), ACTION_LIMIT)
observation, _ = replay_env.reset(seed=REPLAY_SEED)
renderer = mujoco.Renderer(replay_env.model, height=480, width=640)
writer = imageio.get_writer(REPLAY_PATH, fps=REPLAY_FPS, codec="libx264", quality=8)
frame_stride = max(1, round(1 / (replay_env.dt * REPLAY_FPS)))
try:
    for step in range(replay_env.max_episode_steps):
        action, _ = agent.act(observation, deterministic=True)
        observation, _, terminated, truncated, _ = replay_env.step(action)
        if step % frame_stride == 0:
            renderer.update_scene(replay_env.data, camera="replay")
            writer.append_data(renderer.render())
        if terminated or truncated:
            break
finally:
    writer.close()
    renderer.close()
print("Saved replay:", REPLAY_PATH)
display(Video(str(REPLAY_PATH), embed=True))
